In [1]:
# Ipython...
from __future__ import print_function, division
import matplotlib.pyplot as plt
from sympy.physics.vector import init_vprinting
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
get_ipython().run_line_magic('matplotlib', 'inline')
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
init_vprinting(use_latex='mathjax', pretty_print=False)
# Sympy
import numpy as np
import sympy as sp
from sympy import Matrix, symbols, simplify, Function, expand_trig, Symbol, diff
from sympy import cos, sin, transpose, pi
from sympy import latex, python
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, inertia
# Yams sympy
from welib.yams.yams_sympy       import YAMSRigidBody, YAMSInertialBody, YAMSFlexibleBody
from welib.yams.yams_sympy_model import YAMSModel


In [ ]:
# Ipython...
from __future__ import print_function, division
import matplotlib.pyplot as plt
from sympy.physics.vector import init_vprinting
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
get_ipython().run_line_magic('matplotlib', 'inline')
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
init_vprinting(use_latex='mathjax', pretty_print=False)
# Sympy
import numpy as np
import sympy as sp
from sympy import Matrix, symbols, simplify, Function, expand_trig, Symbol, diff
from sympy import cos, sin, transpose, pi
from sympy import latex, python
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, inertia
# Yams sympy
from welib.yams.yams_sympy       import YAMSRigidBody, YAMSInertialBody, YAMSFlexibleBody
from welib.yams.yams_sympy_model import YAMSModel


In [2]:
# -- Degrees of freedom
time = dynamicsymbols._t
theta, psi = dynamicsymbols('theta, psi')
thetad, psid = dynamicsymbols('thetad, psid')

# --- Reference frame
ref = YAMSInertialBody('E') 
nac = YAMSRigidBody('N', rho_G = [Symbol('x_NG'),0,Symbol('z_NG')], J_form='diag') # full, diag, cross
#nac
bld = YAMSRigidBody('B', rho_G = [0,0,Symbol('z_BG')], J_form='diag') # full, diag, cross
bodies=[nac, bld]

# --- Connections
ref.connectTo(nac, type='Joint', rel_pos=(0, 0, 0), rot_type='Body', rot_amounts=(0,theta,0), rot_order='XYZ')
nac.connectTo(bld, type='Joint', rel_pos=(Symbol('x_NB'), 0, Symbol('z_NB')), rot_type='Body', rot_amounts=(psi,0,0), rot_order='XYZ')



In [3]:
# --- DOFs and kinetic equations
coordinates = [theta, psi]
speeds      = [thetad, psid]
kdeqsSubs = []
for q,qd in zip(coordinates, speeds):
    kdeqsSubs += [(qd, sp.diff(q, time))]
print('coordinates:', coordinates)
print('kdeqsSubs  :', kdeqsSubs)

coordinates: [theta(t), psi(t)]
kdeqsSubs  : [(thetad(t), Derivative(theta(t), t)), (psid(t), Derivative(psi(t), t))]


In [4]:
# --- Loads
gravity=Symbol('g')
body_loads = []
for bdy in bodies:
    body_loads  += [(bdy, (bdy.masscenter, -bdy.mass * gravity * ref.frame.z))]  # changed to ref.frame.z from bdy.frame.z to get the correct direction of gravity

    
model = YAMSModel(name='NT', ref=ref, bodies=bodies, body_loads=body_loads, 
                  coordinates=coordinates, speeds=speeds, kdeqsSubs=kdeqsSubs, g_vect=-gravity*ref.frame.z)

model.kaneEquations(Mform='TaylorExpanded')
extraSubs  = []

EOM=model.to_EOM(extraSubs=extraSubs)
EOM.mass_forcing_form()

In [5]:
print('Mass matrix:')
EOM.M
print('Forcing term:')
print(sp.simplify(sp.trigsimp(EOM.F)))

Mass matrix:


Matrix([
[J_yy_B*cos(psi)**2 + J_yy_N + J_zz_B*sin(psi)**2 + M_B*x_NB**2 + M_B*z_BG**2*cos(psi)**2 + 2*M_B*z_BG*z_NB*cos(psi) + M_B*z_NB**2 + M_N*x_NG**2 + M_N*z_NG**2, M_B*x_NB*z_BG*sin(psi)],
[                                                                                                                                       M_B*x_NB*z_BG*sin(psi),   J_xx_B + M_B*z_BG**2]])

Forcing term:
Matrix([[J_yy_B*sin(2*psi(t))*Derivative(psi(t), t)*Derivative(theta(t), t) - J_zz_B*sin(2*psi(t))*Derivative(psi(t), t)*Derivative(theta(t), t) + M_B*g*x_NB*cos(theta(t)) - M_B*g*z_BG*sin(psi(t) - theta(t))/2 + M_B*g*z_BG*sin(psi(t) + theta(t))/2 + M_B*g*z_NB*sin(theta(t)) - M_B*x_NB*z_BG*cos(psi(t))*Derivative(psi(t), t)**2 + M_B*z_BG**2*sin(2*psi(t))*Derivative(psi(t), t)*Derivative(theta(t), t) + 2*M_B*z_BG*z_NB*sin(psi(t))*Derivative(psi(t), t)*Derivative(theta(t), t) + M_N*g*x_NG*cos(theta(t)) + M_N*g*z_NG*sin(theta(t))], [(-J_yy_B*cos(psi(t))*Derivative(theta(t), t)**2 + J_zz_B*cos(psi(t))*Derivative(theta(t), t)**2 + M_B*g*z_BG*cos(theta(t)) - M_B*z_BG**2*cos(psi(t))*Derivative(theta(t), t)**2 - M_B*z_BG*z_NB*Derivative(theta(t), t)**2)*sin(psi(t))]])


In [6]:
M0,C0,K0,F0 = EOM.linearize(noVel=True)
M0
K0

Matrix([
[J_yy_B*cos(psi)**2 + J_yy_N + J_zz_B*sin(psi)**2 + M_B*(x_NB**2 + z_BG**2*cos(psi)**2 + 2*z_BG*z_NB*cos(psi) + z_NB**2) + M_N*(x_NG**2 + z_NG**2), M_B*x_NB*z_BG*sin(psi)],
[                                                                                                                           M_B*x_NB*z_BG*sin(psi),   J_xx_B + M_B*z_BG**2]])

Matrix([
[M_B*g*x_NB*sin(theta) - M_B*g*z_BG*cos(psi)*cos(theta) - M_B*g*z_NB*cos(theta) + M_N*g*x_NG*sin(theta) - M_N*g*z_NG*cos(theta), M_B*g*z_BG*sin(psi)*sin(theta) + M_B*x_NB*z_BG*cos(psi)*Derivative(0, t) + (-2*J_yy_B*sin(psi)*cos(psi) + 2*J_zz_B*sin(psi)*cos(psi) + M_B*(-2*z_BG**2*sin(psi)*cos(psi) - 2*z_BG*z_NB*sin(psi)))*Derivative(0, t)],
[                                                                                                M_B*g*z_BG*sin(psi)*sin(theta),                                                                                                                                          -M_B*g*z_BG*cos(psi)*cos(theta) + M_B*x_NB*z_BG*cos(psi)*Derivative(0, t)]])

In [7]:
#twrA.bodyMassMatrix(form='symbolic')
nac.bodyMassMatrix()

>>> bodyMassMatrix for rigid bodies is in Beta


Matrix([
[     M_N,         0,         0,                    0,                         M_N*z_NG,                    0],
[       0,       M_N,         0,            -M_N*z_NG,                                0,             M_N*x_NG],
[       0,         0,       M_N,                    0,                        -M_N*x_NG,                    0],
[       0, -M_N*z_NG,         0, J_xx_N + M_N*z_NG**2,                                0,       -M_N*x_NG*z_NG],
[M_N*z_NG,         0, -M_N*x_NG,                    0, J_yy_N + M_N*(x_NG**2 + z_NG**2),                    0],
[       0,  M_N*x_NG,         0,       -M_N*x_NG*z_NG,                                0, J_zz_N + M_N*x_NG**2]])